# CME Futures: Sequence Models

This notebook evaluates the declared NLinear and LSTM sequence configurations. Each input window
contains observations from one product and ends before its prediction timestamp. Purge gaps and
fold boundaries prevent a sequence from crossing into another validation interval, and hidden
state does not pass between products or folds.

Every declared epoch checkpoint is published with fitted weights and exact chronological
eligibility. MC dropout is not an undeclared side experiment. Configuration selection remains the
validation backtest decision in `13_backtest`.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures sequence-model population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

The request rows identify architecture, label, and published configuration. Sequence length,
checkpoint schedule, seed, gap policy, and device enter the resolved computation identity.

**The device is declared here rather than inherited.** With no override the shared sequence
adapter falls back to a literal `"cuda"` written in `case_studies/utils/deep_learning.py`, and
resolving the request raises `CUDA was requested for sequence training, but CUDA is unavailable`
rather than quietly moving the fit to the CPU. That refusal comes from resolving the request, so
it arrives before any fitting starts. A CUDA device is therefore a hard requirement of this
population, and stating it in the request puts that requirement where a reader meets it instead
of two layers below. The resolved specification hash is the same with the override as without,
so this names what the published run already did.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog("deep_learning", labels=ALL_LABELS)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": "cuda"},
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""regression""",69,30,26974,5,2019-01-03 00:00:00,2023-11-29 00:00:00,20,"""canonical""","""b3cb11886b68"""
"""deep_learning""","""fwd_ret_21d""","""nlinear""","""regression""",69,30,26974,5,2019-01-03 00:00:00,2023-11-29 00:00:00,20,"""canonical""","""074807be43f1"""
"""deep_learning""","""fwd_ret_5d""","""lstm_h64""","""regression""",69,30,27326,5,2019-01-03 00:00:00,2023-12-21 00:00:00,20,"""canonical""","""eefe7b49664b"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""regression""",69,30,27326,5,2019-01-03 00:00:00,2023-12-21 00:00:00,20,"""canonical""","""ab599c7e590d"""


## Execute and validate

The shared sequence adapter owns window construction, checkpoint reload, prediction coverage, and
restart. A failed configuration cannot remove itself from the population snapshot.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme_futures-deep_learning-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Fold-major CV: 5 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=43,008 seq across 30 symbols
    val=5,078 seq across 30 symbols
    creating datasets...
    datasets ready
    nlinear:


      epoch   1/100: train_loss=0.268148


      epoch   2/100: train_loss=0.094137


      epoch   3/100: train_loss=0.048640


...

      epoch  99/100: train_loss=0.000249


      epoch 100/100: train_loss=0.000248, val_loss=0.003318, IC=+0.1300


      best_ep=60, IC=+0.1409 (145.8s, 20 checkpoints)


  lstm_h64: best_epoch=60, IC=-0.0004 (643.5s)



  Best: lstm_h64 @ epoch 60 (IC=-0.0004)
  Saved to ~/ml4t/public-s6-cme_futures/case_studies/cme_futures/run_log/training/b3cb11886b68/diagnostics


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "config_name", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("sequence execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",5,"""canonical""",true,"""b3cb11886b68""","""cc6ad8b1a3f6"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",10,"""canonical""",true,"""b3cb11886b68""","""9524f0393306"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",15,"""canonical""",true,"""b3cb11886b68""","""8d2d4dd7d585"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",20,"""canonical""",true,"""b3cb11886b68""","""ed07af97228a"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",25,"""canonical""",true,"""b3cb11886b68""","""267a6441a0d1"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",80,"""canonical""",true,"""ab599c7e590d""","""588aaee583ff"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",85,"""canonical""",true,"""ab599c7e590d""","""056ece3612a4"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",90,"""canonical""",true,"""ab599c7e590d""","""2774dc78adc4"""
